# INST326 — Week 9 Exercises: Abstract Classes & Interfaces (Library Management)

**Focus (Week 9 only):** Abstract Base Classes (ABCs) with `abc.ABC` and `@abstractmethod`, abstract properties, virtual subclass registration, and light “interface-like” design via ABCs and (optional) `typing.Protocol` **without** advanced generics.

**Out of scope (Week 10+):** multiple inheritance/mixins, advanced design patterns, dependency injection, metaclasses beyond `ABCMeta`, decorators beyond basics, complex type-system features (ParamSpec, TypeVar variance), context managers beyond prior weeks.


### Starter Scaffold (Week-9-safe)

Below is a minimal domain model from prior weeks, slightly adapted for Week 9. We keep inheritance simple and introduce **abstract classes** to define common contracts.


In [3]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Protocol
from abc import ABC, abstractmethod

# --- Exceptions (kept simple) ---
class LibraryError(Exception): ...
class DuplicateBookError(LibraryError): ...
class OverdueLoanError(LibraryError): ...
class NonBorrowableError(LibraryError): ...

# --- Abstract base for library items ---
class LibraryItem(ABC):
    def __init__(self, isbn: str, title: str, copies: int = 1) -> None:
        self.isbn = isbn
        self.title = title
        self.copies = copies

    @abstractmethod
    def loan_period_days(self) -> int:
        """Each concrete item defines its loan period."""

    @abstractmethod
    def describe(self) -> str:
        """Human-readable description of the item."""

    @property
    @abstractmethod
    def is_digital(self) -> bool:
        """Whether the item is digital."""

    # A partial template method that depends on an abstract method.
    def due_date_from_today(self) -> datetime:
        return datetime.now() + timedelta(days=self.loan_period_days())

    # default stock policy common to most items
    def can_checkout(self) -> bool:
        return self.copies > 0

# --- Concrete items (will be extended in exercises) ---
class PrintedBook(LibraryItem):
    def loan_period_days(self) -> int:
        return 21
    def describe(self) -> str:
        return f"PrintedBook<{self.isbn}>: {self.title} (copies={self.copies})"
    @property
    def is_digital(self) -> bool:
        return False

class EBook(LibraryItem):
    def __init__(self, isbn: str, title: str, copies: int = 0, file_size_mb: float = 0.0) -> None:
        super().__init__(isbn, title, copies)
        self.file_size_mb = float(file_size_mb)
    def loan_period_days(self) -> int:
        return 14
    def describe(self) -> str:
        return f"EBook<{self.isbn}>: {self.title} ({self.file_size_mb:.1f} MB)"
    @property
    def is_digital(self) -> bool:
        return True
    # Different stock rule (licenses can be zero but not negative)
    def can_checkout(self) -> bool:
        return self.copies >= 0

class AudioBook(LibraryItem):
    def __init__(self, isbn: str, title: str, copies: int = 1, duration_min: int = 0) -> None:
        super().__init__(isbn, title, copies)
        self.duration_min = int(duration_min)
    def loan_period_days(self) -> int:
        return 14
    def describe(self) -> str:
        return f"AudioBook<{self.isbn}>: {self.title} ({self.duration_min} min)"
    @property
    def is_digital(self) -> bool:
        return True

# --- Optional Protocol example (kept simple & non-generic) ---
class Downloadable(Protocol):
    def download_link(self) -> str: ...

# --- Core containers ---
@dataclass
class Member:
    member_id: str
    email: str
    def max_concurrent_loans(self) -> int:
        return 5

@dataclass
class Loan:
    isbn: str
    member_id: str
    due_date: datetime
    returned: bool = False
    def mark_returned(self) -> None:
        self.returned = True

class Catalog:
    def __init__(self):
        self._items: Dict[str, LibraryItem] = {}
    def add_item(self, item: LibraryItem) -> None:
        if item.isbn in self._items:
            raise DuplicateBookError(f"ISBN already exists: {item.isbn}")
        if item.copies < 0:
            raise ValueError("copies must be non-negative")
        self._items[item.isbn] = item
    def get_item(self, isbn: str) -> Optional[LibraryItem]:
        return self._items.get(isbn)

class LoanDesk:
    def __init__(self, catalog: Catalog):
        self.catalog = catalog
        self.loans: List[Loan] = []
    def active_loans_for(self, member: Member) -> List[Loan]:
        return [L for L in self.loans if (L.member_id == member.member_id and not L.returned)]
    def checkout(self, member: Member, item: LibraryItem) -> Loan:
        if len(self.active_loans_for(member)) >= member.max_concurrent_loans():
            raise LibraryError("concurrent loan limit reached")
        if not item.can_checkout():
            raise NonBorrowableError("cannot checkout under current stock policy")
        item.copies -= 1
        loan = Loan(isbn=item.isbn, member_id=member.member_id, due_date=item.due_date_from_today())
        self.loans.append(loan)
        return loan
    def checkin(self, loan: Loan) -> None:
        if not loan.returned:
            it = self.catalog.get_item(loan.isbn)
            if it:
                it.copies += 1
            loan.mark_returned()

@dataclass
class Member:
    member_id: str
    email: str
    role: MemberRole = None
    
    def __post_init__(self):
        if self.role is None:
            self.role = StudentRole()
    
    def max_concurrent_loans(self) -> int:
        return self.role.max_concurrent_loans()

@dataclass
class Loan:
    isbn: str
    member_id: str
    due_date: datetime
    returned: bool = False
    def mark_returned(self) -> None:
        self.returned = True

class Catalog:
    def __init__(self):
        self._items: Dict[str, LibraryItem] = {}
    def add_item(self, item: LibraryItem) -> None:
        if item.isbn in self._items:
            raise DuplicateBookError(f"ISBN already exists: {item.isbn}")
        item.validate_on_add()
        self._items[item.isbn] = item
    def get_item(self, isbn: str) -> Optional[LibraryItem]:
        return self._items.get(isbn)



## 1) Make `LibraryItem` truly abstract

Prove that `LibraryItem` cannot be instantiated directly. Write a quick try/except demonstrating that creating `LibraryItem('x','y')` raises a `TypeError` because of abstract methods.

In [55]:
# Your code here
# Demonstrate TypeError on instantiation of abstract class
class LibraryItem(ABC):
    def __init__(self, isbn: str, title: str, copies: int = 1) -> None:
        self.isbn = isbn
        self.title = title
        self.copies = copies

    @abstractmethod
    def loan_period_days(self) -> int:
        """Each concrete item defines its loan period."""

    @abstractmethod
    def describe(self) -> str:
        """Human-readable description of the item."""

    @property
    @abstractmethod
    def is_digital(self) -> bool:
        """Whether the item is digital."""

    #demo
    def demo_exercise_1():
        print("=== Exercise 1: Abstract Class Instantiation ===")
    try:
        item = LibraryItem('123', 'Test Book')
        print("ERROR: Should not reach here!")
    except TypeError as e:
        print(f"✓ Cannot instantiate abstract class: {type(e).__name__}")
    print()

✓ Cannot instantiate abstract class: TypeError



## 2) Abstract property practice

Add an **abstract property** `media_type` to `LibraryItem` and implement it in all concrete subclasses with strings like `'print'`, `'ebook'`, `'audiobook'`.

In [5]:
# Your code here
# Add @property @abstractmethod def media_type(self) -> str: ...
@property
@abstractmethod
def media_type(self) -> str:
        """Type of media (print, ebook, audiobook, etc.)"""

## 3) Abstract classmethod

Add an abstract `@classmethod def kind(cls) -> str` to `LibraryItem` returning a short identifier (e.g., `'book'`). Implement it for the concrete subclasses.

In [6]:
# Your code here
# classmethod kind() -> str on LibraryItem and overrides
@classmethod
@abstractmethod
def kind(cls) -> str:
    """Return the kind of library item."""

## 4) Template method using abstract hook

Create a template method `receipt_line(self) -> str` on `LibraryItem` that uses `self.describe()` and `self.loan_period_days()`. Show that each subclass inherits the same template but outputs different text due to overrides.

In [7]:
# Your code here
# Implement receipt_line in LibraryItem using abstract hooks
def receipt_line(self) -> str:
    return f"{self.title} ({self.kind()}) - Due: {self.due_date_from_today().date()}"


## 5) Virtual subclass registration

Create a new class `PDFPamphlet` **without** inheriting from `LibraryItem`, but `register` it as a virtual subclass using `LibraryItem.register(PDFPamphlet)`. Implement the required interface manually. Show that `isinstance(pdf, LibraryItem)` returns `True` after registration.

In [ ]:
# Your code here
# Define PDFPamphlet, register as virtual subclass, demonstrate isinstance
class PDFPamphlet:
    def __init__(self, isbn: str, title: str, file_size_mb: float = 0.0) -> None:
        self.isbn = isbn
        self.title = title
        self.file_size_mb = float(file_size_mb)

    def loan_period_days(self) -> int:
        return 7

    def describe(self) -> str:
        return f"PDFPamphlet<{self.isbn}>: {self.title} ({self.file_size_mb:.1f} MB)"

    @property
    def is_digital(self) -> bool:
        return True

    @classmethod
    def kind(cls) -> str:
        return "PDF Pamphlet"

## 6) Abstract property for availability

Add an abstract property `borrowable: bool` to `LibraryItem`. For `PrintedBook` and `AudioBook`, return `True`. For `EBook`, return `True` if `copies >= 0`. Demonstrate a check before `LoanDesk.checkout` that raises `NonBorrowableError` if `borrowable` is False.

In [65]:
# Your code here
# Add property and demonstrate guarding behavior
@property
@abstractmethod
def borrowable(self) -> bool:
        """Whether this item can be borrowed."""
# Test with non-borrowable item (create one with negative copies to trigger error)
try:
    non_borrowable = PrintedBook('NB006', 'No Copies', copies=0)
    non_borrowable.copies = 0  # Set to 0 to make it non-checkable
    print(f"\nPrintedBook with 0 copies - borrowable: {non_borrowable.borrowable}")
    print(f"PrintedBook with 0 copies - can_checkout: {non_borrowable.can_checkout()}")
except NonBorrowableError as e:
    print(f"✓ NonBorrowableError raised: {e}")


PrintedBook with 0 copies - borrowable: True
PrintedBook with 0 copies - can_checkout: False


## 7) EBook implements Downloadable Protocol

Implement `download_link(self) -> str` on `EBook` to satisfy `Downloadable`. Write a function `offer_download(x)` that accepts a `Downloadable` and returns its link. Show duck-typed use with an `EBook` instance.

In [56]:
# Your code here
# EBook.download_link and offer_download(downloadable)
def download_link(self) -> str:
    return f"https://library.example.com/download/{self.isbn}"
def offer_download(downloadable: Downloadable) -> str:
    return downloadable.download_link()
# Register PDFPamphlet as a virtual subclass of LibraryItem
LibraryItem.register(PDFPamphlet)

#demo
def demo_exercise_7():
    """Exercise 7: Downloadable protocol."""
    print("=== Exercise 7: Downloadable Protocol ===")
    ebook = EBook('EB001', 'Digital Future', copies=100, file_size_mb=5.2)
    link = offer_download(ebook)
    print(f"Download link: {link}")
    print()

## 8) Protocol vs ABC (short reflection)

In a short markdown cell, explain the difference between using an ABC and a Protocol for “interfaces” in Python, and when you might pick one over the other in this project.

In [ ]:
# (Write your reflection in this markdown cell.)
# Reflection: The difference between using an ABC and a Protocol for interfaces
# is that ABCs enforce a inheritance factor, but protocols allow me flexiablity


## 9) `LoanDesk` typed for abstraction

Refactor type hints in `LoanDesk` to accept `LibraryItem` rather than concrete classes everywhere. Explain (markdown) why depending on the abstract type improves flexibility.

In [10]:
# Your code here
# (Minor changes may already reflect this in the scaffold.)
# Implement media_type in concrete classes
def demonstrate_loan_desk_flexibility():
    """Shows LoanDesk.checkout accepts any LibraryItem subclass."""
    test_catalog = Catalog()
    test_desk = LoanDesk(test_catalog)
    test_member = Member('M999', 'flex@test.com')
    
    various_items = [
        PrintedBook('FLEX1', 'Book', copies=1),
        EBook('FLEX2', 'EBook', copies=1),
        AudioBook('FLEX3', 'Audio', copies=1),
        PDFPamphlet('FLEX4', 'PDF', copies=1),
        FixedItem('FLEX5', 'Fixed', copies=1)
    ]
    
    for item in various_items:
        test_catalog.add_item(item)
        loan = test_desk.checkout(test_member, item)
        print(f"Exercise 9: Checked out {type(item).__name__} via LibraryItem interface")
    
    return test_desk

## 10) Abstract fee policy

Add an abstract method `daily_late_fee(self) -> float` to `LibraryItem`. Implement fees:
- PrintedBook: 0.25
- EBook: 0.10
- AudioBook: 0.15
Add a concrete `late_fee(self, days_late: int) -> float` in `LibraryItem` that multiplies days by `daily_late_fee()`.

In [11]:
# Your code here
# Add abstract daily_late_fee and concrete late_fee to LibraryItem and overrides
@abstractmethod
def daily_late_fee(self) -> float:
        """Daily late fee for this item type."""
def late_fee(self, days_late: int) -> float:
        return max(0, days_late) * self.daily_late_fee()
# Example implementation in PrintedBook
def daily_late_fee(self) -> float:
        return 0.25
PrintedBook.daily_late_fee = daily_late_fee
PrintedBook.late_fee = late_fee


## 11) ABC for Member roles

Create an abstract base `MemberRole(ABC)` with `max_concurrent_loans(self) -> int`. Implement `StudentRole` (5) and `StaffRole` (10). Modify `Member` to hold a `role: MemberRole` and delegate `max_concurrent_loans()` to it. Keep the implementation single-inheritance (no mixins).

In [12]:
# Your code here
# Define roles and update Member
class Role(Protocol):
    def max_concurrent_loans(self) -> int: ...
class StudentRole:
    def max_concurrent_loans(self) -> int:
        return 7
class FacultyRole:
    def max_concurrent_loans(self) -> int:
        return 10

## 12) Prevent partial implementations

Create a subclass `BrokenItem(LibraryItem)` that **forgets** to implement one abstract member. Show that instantiating it raises `TypeError`. Then fix it by implementing the missing member.

In [ ]:
# Your code here
# Show failing instantiation then the fix
class FixedItem(LibraryItem):
    def loan_period_days(self) -> int:
        return 0
    def describe(self) -> str:
        return f"FixedItem<{self.isbn}>: {self.title} (not borrowable)"
    @property
    def is_digital(self) -> bool:
        return False
    def can_checkout(self) -> bool:
        return False

## 13) Abstract validation hook

Add an abstract hook `validate_on_add(self) -> None` to `LibraryItem` and override it in each subclass to enforce a simple constraint (e.g., `copies >= 0`). Modify `Catalog.add_item` to call `item.validate_on_add()` before insertion.

In [17]:
# Your code here
# Add validate_on_add to classes and call from Catalog.add_item
def validate_on_add(self) -> None:
        if self.copies < 0:
            raise ValueError("copies must be non-negative")
PrintedBook.validate_on_add = validate_on_add
EBook.validate_on_add = validate_on_add
AudioBook.validate_on_add = validate_on_add
print("Exercise 13: Added validate_on_add to concrete classes")

Exercise 11: Added validate_on_add to concrete classes


## 14) Minimal adapter via ABC registration

Suppose you receive third-party objects with attributes `code`, `name`, `stock` that you want to treat as `LibraryItem`. Write a light Adapter class that **implements** the `LibraryItem` API and delegates to the 3rd-party object, then register it (or the 3rd-party class) appropriately. Show it working with `LoanDesk.checkout`.

In [15]:
# Your code here
# Implement Adapter that conforms to LibraryItem contract
class ThirdPartyBook:
    def __init__(self, code: str, name: str, stock: int):
        self.code = code
        self.name = name
        self.stock = stock

class ThirdPartyAdapter(LibraryItem):
    def __init__(self, third_party_obj: ThirdPartyBook):
        self._obj = third_party_obj
        super().__init__(isbn=third_party_obj.code, title=third_party_obj.name, 
                        copies=third_party_obj.stock)
    def loan_period_days(self) -> int:
        return 14
    def describe(self) -> str:
        return f"ThirdParty<{self.isbn}>: {self.title}"
    @property
    def is_digital(self) -> bool:
        return False
    @property
    def media_type(self) -> str:
        return 'print'
    @classmethod
    def kind(cls) -> str:
        return 'book'
    @property
    def borrowable(self) -> bool:
        return True
    def daily_late_fee(self) -> float:
        return 0.20
    def validate_on_add(self) -> None:
        if self.copies < 0:
            raise ValueError("copies must be non-negative")

third_party = ThirdPartyBook('TP001', 'External Book', 3)
adapted = ThirdPartyAdapter(third_party)
print(f"Exercise 14: Adapted book: {adapted.describe()}")

Exercise 14: Adapted book: ThirdParty<TP001>: External Book


## 15) Unit test: abstract contract

Using `unittest`, write tests that assert:
- `LibraryItem` instantiation fails
- All concrete classes implement `loan_period_days` and `describe`
- `late_fee` uses the subclass-specific `daily_late_fee` values

In [18]:
# Your code here
import unittest

class TestWeek9ABCs(unittest.TestCase):
    def test_abstract_instantiation(self):
        ...
    def test_concretes_implement_contract(self):
        ...
    def test_late_fee_polymorphism(self):
        ...

# # To run tests in notebook:
class TestWeek9ABCs(unittest.TestCase):
    def test_abstract_instantiation(self):
        """Test that LibraryItem cannot be instantiated directly."""
        with self.assertRaises(TypeError):
            LibraryItem('123', 'Test')
    
    def test_concretes_implement_contract(self):
        """Test that all concrete classes implement loan_period_days and describe."""
        book = PrintedBook('001', 'Python Basics')
        ebook = EBook('002', 'Advanced Python', copies=5, file_size_mb=2.5)
        audio = AudioBook('003', 'Learn by Listening', duration_min=300)
        
        # Test that all concrete classes implement required abstract methods
        for item in [book, ebook, audio]:
            # loan_period_days returns int
            self.assertIsInstance(item.loan_period_days(), int)
            self.assertGreater(item.loan_period_days(), 0)
            
            # describe returns str
            self.assertIsInstance(item.describe(), str)
            self.assertIn(item.isbn, item.describe())
            
            # is_digital returns bool
            self.assertIsInstance(item.is_digital, bool)
            
            # media_type returns str
            self.assertIsInstance(item.media_type, str)
            self.assertIn(item.media_type, ['print', 'ebook', 'audiobook'])
            
            # kind returns str
            self.assertIsInstance(item.kind(), str)
            
            # borrowable returns bool
            self.assertIsInstance(item.borrowable, bool)
    
    def test_late_fee_polymorphism(self):
        """Test that late_fee uses the subclass-specific daily_late_fee values."""
        book = PrintedBook('001', 'Test')
        ebook = EBook('002', 'Test')
        audio = AudioBook('003', 'Test')
        
        # Test daily_late_fee values
        self.assertEqual(book.daily_late_fee(), 0.25)
        self.assertEqual(ebook.daily_late_fee(), 0.10)
        self.assertEqual(audio.daily_late_fee(), 0.15)
        
        # Test late_fee calculation (10 days late)
        self.assertEqual(book.late_fee(10), 2.50)   # 10 * 0.25
        self.assertEqual(ebook.late_fee(10), 1.00)  # 10 * 0.10
        self.assertEqual(audio.late_fee(10), 1.50)  # 10 * 0.15
        
        # Test with different days
        self.assertEqual(book.late_fee(5), 1.25)    # 5 * 0.25
        self.assertEqual(ebook.late_fee(20), 2.00)  # 20 * 0.10
        self.assertEqual(audio.late_fee(1), 0.15)   # 1 * 0.15

print("Exercise 15: Unit tests defined - run unittest.main() to execute")

# # unittest.main(argv=['-v'], exit=False)

Exercise 15: Unit tests defined - run unittest.main() to execute


## 16) Swap implementation behind the ABC

Write a function `checkout_any(desk: LoanDesk, member: Member, item: LibraryItem)` that works for **any** `LibraryItem` or registered virtual subclass. Demonstrate with a `PDFPamphlet` instance (from Ex. 5) and a normal `PrintedBook`.

In [37]:
# Your code here
def checkout_any(desk: LoanDesk, member: Member, item: LibraryItem) -> Loan:
    return desk.checkout(member, item)

catalog = Catalog()
desk = LoanDesk(catalog)
member = Member('M003', 'test@ex.com')

catalog.add_item(book)
catalog.add_item(adapted)  # Use the adapted instance  # noqa: F821 # Use the adapted instance
print(f"Exercise 16: Checked out {len(desk.active_loans_for(member))} items")

Exercise 16: Checked out 0 items


## 17) Abstract property + computed template

Add a template method `full_label()` to `LibraryItem` that returns `f"[{self.media_type}] {self.title} — {self.isbn}"`. Confirm each subclass inherits it and shows the right `media_type` value.

In [52]:
def full_label(self) -> str:
        """Full label with media type, title, and ISBN."""
        return f"[{self.media_type}] {self.title} – {self.isbn}"

# --- Concrete items ---
class PrintedBook(LibraryItem):
    def loan_period_days(self) -> int:
        return 21
    def describe(self) -> str:
        return f"PrintedBook<{self.isbn}>: {self.title} (copies={self.copies})"
    @property
    def is_digital(self) -> bool:
        return False
    @property
    def media_type(self) -> str:
        return 'print'
    @classmethod
    def kind(cls) -> str:
        return 'book'
    @property
    def borrowable(self) -> bool:
        return True
    def daily_late_fee(self) -> float:
        return 0.25
    def validate_on_add(self) -> None:
        if self.copies < 0:
            raise ValueError("PrintedBook copies must be non-negative")

class EBook(LibraryItem):
    def __init__(self, isbn: str, title: str, copies: int = 0, file_size_mb: float = 0.0) -> None:
        super().__init__(isbn, title, copies)
        self.file_size_mb = float(file_size_mb)
    def loan_period_days(self) -> int:
        return 14
    def describe(self) -> str:
        return f"EBook<{self.isbn}>: {self.title} ({self.file_size_mb:.1f} MB)"
    @property
    def is_digital(self) -> bool:
        return True
    @property
    def media_type(self) -> str:
        return 'ebook'
    @classmethod
    def kind(cls) -> str:
        return 'book'
    @property
    def borrowable(self) -> bool:
        return self.copies >= 0
    def daily_late_fee(self) -> float:
        return 0.10
    def validate_on_add(self) -> None:
        if self.copies < 0:
            raise ValueError("EBook copies must be non-negative")
    def can_checkout(self) -> bool:
        return self.copies >= 0
    class AudioBook(LibraryItem):
        def __init__(self, isbn: str, title: str, copies: int = 1, duration_min: int = 0) -> None:
            super().__init__(isbn, title, copies)
            self.duration_min = int(duration_min)
        
        def loan_period_days(self) -> int:
            return 14
        
        def describe(self) -> str:
            return f"AudioBook<{self.isbn}>: {self.title} ({self.duration_min} min)"
        
        @property
        def is_digital(self) -> bool:
            return True
        
        @property
        def media_type(self) -> str:
            return 'audiobook'
        
        @classmethod
        def kind(cls) -> str:
            return 'book'
        
        @property
        def borrowable(self) -> bool:
            return True
        
        def daily_late_fee(self) -> float:
            return 0.15
        
        def validate_on_add(self) -> None:
            if self.copies < 0:
                raise ValueError("AudioBook copies must be non-negative")

PrintedBook.full_label = full_label
EBook.full_label = full_label
AudioBook.full_label = full_label
PDFPamphlet.full_label = full_label
    



## 18) issubclass / isinstance with ABCs

Show examples of `issubclass(PrintedBook, LibraryItem)` and `isinstance(EBook(...), LibraryItem)`. After registering a virtual subclass, show `issubclass(PDFPamphlet, LibraryItem)` is `True` as well.

In [59]:
print("\nExercise 18: issubclass/isinstance with ABCs")

# Show issubclass for real subclasses
print(f"issubclass(PrintedBook, LibraryItem): {issubclass(PrintedBook, LibraryItem)}")
print(f"issubclass(EBook, LibraryItem): {issubclass(EBook, LibraryItem)}")
print(f"issubclass(AudioBook, LibraryItem): {issubclass(AudioBook, LibraryItem)}")

# Show isinstance for objects
book18 = PrintedBook('ISBN18', 'Test Book')
ebook18 = EBook('EISBN18', 'Test EBook')
print(f"\nisinstance(book18, LibraryItem): {isinstance(book18, LibraryItem)}")
print(f"isinstance(ebook18, LibraryItem): {isinstance(ebook18, LibraryItem)}")

# After registering PDFPamphlet as virtual subclass
print(f"\nVirtual subclass registration:")
print(f"issubclass(PDFPamphlet, LibraryItem): {issubclass(PDFPamphlet, LibraryItem)}")
pdf18 = PDFPamphlet('PISBN18', 'Test PDF')
print(f"isinstance(pdf18, LibraryItem): {isinstance(pdf18, LibraryItem)}")
print(f"Note: PDFPamphlet does NOT inherit from LibraryItem, but is registered!")


Exercise 18: issubclass/isinstance with ABCs
issubclass(PrintedBook, LibraryItem): True
issubclass(EBook, LibraryItem): True
issubclass(AudioBook, LibraryItem): True

isinstance(book18, LibraryItem): True
isinstance(ebook18, LibraryItem): True

Virtual subclass registration:
issubclass(PDFPamphlet, LibraryItem): True
isinstance(pdf18, LibraryItem): True
Note: PDFPamphlet does NOT inherit from LibraryItem, but is registered!


## 19) Inventory report via abstraction

Write `summarize_items(items: list[LibraryItem]) -> list[str]` that uses only abstract methods/properties (`describe`, `media_type`, etc.). Demonstrate polymorphism by passing a mixed list of items.

In [60]:
# Your code here
def summarize_items(items: list[LibraryItem]) -> list[str]:
    return [f"{item.full_label()} - {item.describe()}" for item in items]

items = [book, ebook, audio, pdf]
summary = summarize_items(items)
print("Exercise 19: Inventory Summary")
for line in summary:
    print(f"  {line}")
    ...

Exercise 19: Inventory Summary
  [print] Python Guide – PB001 - PrintedBook<PB001>: Python Guide (copies=3)
  [ebook] Web Dev – EB001 - EBook<EB001>: Web Dev (3.2 MB)
  [audiobook] Learn Python – AB001 - AudioBook<AB001>: Learn Python (180 min)
  [pdf] Quick Guide – PDF001 - PDFPamphlet<PDF001>: Quick Guide


## 20) End-to-end scenario under ABC contract

Create a demo that:
- Builds a `Catalog` and `LoanDesk`
- Adds one of each concrete item and one registered virtual subclass instance
- Checks each out to a `Member`
- Prints a small receipt using `receipt_line()` and the computed due dates
Use only `LibraryItem`-level APIs at call sites (no `isinstance` branches).

In [61]:
# Your code here
# End-to-end demo using the abstract contract only
catalog2 = Catalog()
desk2 = LoanDesk(catalog2)
member2 = Member('M004', 'demo@ex.com', StudentRole())

demo_items = [
    PrintedBook('PB100', 'Python Programming', copies=3),
    EBook('EB100', 'Web Development', copies=10, file_size_mb=4.5),
    AudioBook('AB100', 'Data Science', copies=2, duration_min=180),
    PDFPamphlet('PDF100', 'Quick Reference', copies=5)
]

for item in demo_items:
    catalog2.add_item(item)

print("\nExercise 20: End-to-End Demo")
for item in demo_items:
    loan = desk2.checkout(member2, item)
    print(f"  {item.receipt_line()}")
    print(f"    Due: {loan.due_date.strftime('%Y-%m-%d')}")

print(f"\nTotal active loans: {len(desk2.active_loans_for(member2))}")

# Run unit tests
if __name__ == '__main__':
    unittest.main(argv=[''], exit=False, verbosity=2)

test_abstract_instantiation (__main__.TestWeek9ABCs)
Test that LibraryItem cannot be instantiated directly. ... ok
test_concretes_implement_contract (__main__.TestWeek9ABCs)
Test that all concrete classes implement loan_period_days and describe. ... ok
test_late_fee_polymorphism (__main__.TestWeek9ABCs)
Test that late_fee uses the subclass-specific daily_late_fee values. ... 


Exercise 20: End-to-End Demo
  PrintedBook<PB100>: Python Programming (copies=2) - Due in 21 days
    Due: 2025-12-07
  EBook<EB100>: Web Development (4.5 MB) - Due in 14 days
    Due: 2025-11-30
  AudioBook<AB100>: Data Science (180 min) - Due in 14 days
    Due: 2025-11-30
  PDFPamphlet<PDF100>: Quick Reference - Due in 7 days
    Due: 2025-11-23

Total active loans: 4


ok

----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK


## Python skills you'll need (Weeks 1–9)

- **Core syntax & data types:** variables, strings, numbers, booleans
- **Collections:** lists, dicts (basic use), simple comprehensions
- **Control flow:** `if/elif/else`, `for`, `while`
- **Functions & modules:** defining functions, parameters, returns, imports
- **File I/O & JSON (basic):** open/read/write, simple JSON usage
- **Classes & objects (Weeks 4–8):** classes, `__init__`, instance methods, overriding, `super()`
- **Encapsulation basics:** simple validation; naming conventions for "private" attributes
- **Error handling & testing (Week 7):** `try/except`, custom exceptions, basic `unittest`
- **Week 8 OOP:** single inheritance & polymorphism (no ABCs)
- **Week 9 focus:** **Abstract Base Classes (ABC)** with `abc.ABC` and `@abstractmethod`, abstract properties, class/instance abstract methods, virtual subclass **registration**, and light `typing.Protocol` usage (non-generic)
- **Standard library familiarity:** `abc`, `datetime`, built-in exceptions
